In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# อัลกอริทึมการหาค่าเหมาะสมแบบควอนตัมโดยประมาณ

*ประมาณเวลาการใช้งาน: 22 นาทีบนโปรเซสเซอร์ Heron r3 (หมายเหตุ: นี่เป็นเพียงการประมาณเท่านั้น เวลาจริงอาจแตกต่างกัน)*
## พื้นหลัง
บทแนะนำนี้แสดงวิธีการนำ **อัลกอริทึมการหาค่าเหมาะสมแบบควอนตัมโดยประมาณ (Quantum Approximate Optimization Algorithm หรือ QAOA)** มาใช้งาน ซึ่งเป็นวิธีการแบบไฮบริด (ควอนตัม-คลาสสิก) แบบวนซ้ำ ภายในบริบทของ Qiskit patterns โดยจะเริ่มจากการแก้ปัญหา **Maximum-Cut** (หรือ **Max-Cut**) สำหรับกราฟขนาดเล็ก แล้วเรียนรู้วิธีรันในระดับ utility scale ทุกการรันบนฮาร์ดแวร์ในบทแนะนำนี้ควรอยู่ภายในขีดจำกัดเวลาของ Open Plan ที่เข้าถึงได้ฟรี

ปัญหา Max-Cut เป็นปัญหาการหาค่าเหมาะสมที่ยากต่อการแก้ (โดยเฉพาะคือเป็นปัญหา *NP-hard*) และมีการประยุกต์ใช้หลายอย่างในด้านการจัดกลุ่ม วิทยาศาสตร์เครือข่าย และฟิสิกส์สถิติ บทแนะนำนี้พิจารณากราฟของโหนดที่เชื่อมต่อกันด้วยเส้นเชื่อม และมีเป้าหมายเพื่อแบ่งโหนดออกเป็นสองกลุ่มเพื่อให้จำนวนเส้นเชื่อมที่ผ่านการตัดนี้มีค่าสูงสุด

![ภาพประกอบของปัญหา max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## ข้อกำหนด
ก่อนเริ่มบทแนะนำนี้ ตรวจสอบให้แน่ใจว่าคุณได้ติดตั้งสิ่งต่อไปนี้แล้ว:
- Qiskit SDK v1.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)

นอกจากนี้คุณจะต้องมีสิทธิ์เข้าถึง instance บน [IBM Quantum Platform](/guides/cloud-setup) โปรดทราบว่าบทแนะนำนี้ไม่สามารถรันบน Open Plan ได้ เนื่องจากรันงานโดยใช้ [sessions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session) ซึ่งใช้ได้เฉพาะกับ Premium Plan เท่านั้น
## การตั้งค่า

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## ส่วนที่ 1: QAOA ขนาดเล็ก
ส่วนแรกของบทแนะนำนี้ใช้ปัญหา Max-Cut ขนาดเล็กเพื่ออธิบายขั้นตอนการแก้ปัญหาการหาค่าเหมาะสมโดยใช้คอมพิวเตอร์ควอนตัม

เพื่อให้เข้าใจบริบทก่อนที่จะนำปัญหานี้ไปแมปกับอัลกอริทึมควอนตัม เราสามารถเข้าใจได้ดีขึ้นว่าปัญหา Max-Cut กลายเป็นปัญหาการหาค่าเหมาะสมเชิงคอมบินาทอริกลาสสิกอย่างไร โดยเริ่มจากการพิจารณาการหาค่าต่ำสุดของฟังก์ชัน $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

โดยที่ input $x$ คือเวกเตอร์ที่แต่ละองค์ประกอบสอดคล้องกับแต่ละโหนดของกราฟ จากนั้นกำหนดให้แต่ละองค์ประกอบเหล่านี้เป็น $0$ หรือ $1$ (ซึ่งแทนการรวมอยู่หรือไม่รวมอยู่ในการตัด) ตัวอย่างขนาดเล็กนี้ใช้กราฟที่มี $n=5$ โหนด

คุณสามารถเขียนฟังก์ชันของคู่โหนด $i,j$ ที่บอกว่าเส้นเชื่อม $(i,j)$ อยู่ในการตัดหรือไม่ ตัวอย่างเช่น ฟังก์ชัน $x_i + x_j - 2 x_i x_j$ มีค่าเป็น 1 เมื่อ $x_i$ หรือ $x_j$ ตัวใดตัวหนึ่งเป็น 1 (หมายความว่าเส้นเชื่อมอยู่ในการตัด) และเป็น 0 ในกรณีอื่น ปัญหาในการหาค่าสูงสุดของเส้นเชื่อมในการตัดสามารถกำหนดเป็น

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

ซึ่งสามารถเขียนใหม่ในรูปการหาค่าต่ำสุดได้เป็น

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

ค่าต่ำสุดของ $f(x)$ ในกรณีนี้คือเมื่อจำนวนเส้นเชื่อมที่ผ่านการตัดมีค่าสูงสุด ดังที่เห็น ยังไม่มีส่วนใดที่เกี่ยวข้องกับการคำนวณควอนตัม คุณต้องกำหนดปัญหาใหม่ให้เป็นสิ่งที่คอมพิวเตอร์ควอนตัมเข้าใจได้
เริ่มต้นปัญหาโดยสร้างกราฟที่มี $n=5$ โหนด

In [2]:
n = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### ขั้นตอนที่ 1: แมป input คลาสสิกไปยังปัญหาควอนตัม
ขั้นตอนแรกของ pattern คือการแมปปัญหาคลาสสิก (กราฟ) ไปเป็น Circuit และ operator ควอนตัม ในการทำเช่นนี้มีสามขั้นตอนหลัก:

1. ใช้ชุดของการกำหนดนิยามทางคณิตศาสตร์ใหม่ เพื่อแทนปัญหานี้โดยใช้สัญลักษณ์ของปัญหา Quadratic Unconstrained Binary Optimization (QUBO)
2. เขียนปัญหาการหาค่าเหมาะสมใหม่เป็น Hamiltonian ที่ ground state สอดคล้องกับคำตอบที่หาค่าต่ำสุดของฟังก์ชันต้นทุน
3. สร้าง Circuit ควอนตัมที่จะเตรียม ground state ของ Hamiltonian นี้ผ่านกระบวนการที่คล้ายกับ quantum annealing

**หมายเหตุ:** ในระเบียบวิธี QAOA สิ่งที่ต้องการในท้ายที่สุดคือ operator (**Hamiltonian**) ที่แทน **ฟังก์ชันต้นทุน** ของอัลกอริทึมไฮบริดของเรา รวมถึง Circuit แบบ parametrized (**Ansatz**) ที่แทนสถานะควอนตัมพร้อมคำตอบที่เป็นตัวเลือก คุณสามารถสุ่มตัวอย่างจากสถานะที่เป็นตัวเลือกเหล่านี้ แล้วประเมินโดยใช้ฟังก์ชันต้นทุน

#### กราฟ &rarr; ปัญหาการหาค่าเหมาะสม
ขั้นตอนแรกของการแมปคือการเปลี่ยนสัญลักษณ์ ต่อไปนี้แสดงปัญหาในสัญลักษณ์ QUBO:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

โดยที่ $Q$ คือเมทริกซ์ขนาด $n\times n$ ของจำนวนจริง $n$ สอดคล้องกับจำนวนโหนดในกราฟ $x$ คือเวกเตอร์ของตัวแปรไบนารีที่แนะนำไว้ข้างต้น และ $x^T$ หมายถึง transpose ของเวกเตอร์ $x$

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### ปัญหาการหาค่าเหมาะสม &rarr; Hamiltonian
จากนั้นคุณสามารถกำหนดปัญหา QUBO ใหม่เป็น **Hamiltonian** (ที่นี่คือเมทริกซ์ที่แทนพลังงานของระบบ):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**ขั้นตอนการกำหนดนิยามใหม่จากปัญหา QAOA ไปยัง Hamiltonian**
</summary>

เพื่อแสดงให้เห็นว่าปัญหา QAOA สามารถเขียนใหม่ได้อย่างไร ให้แทนที่ตัวแปรไบนารี $x_i$ ด้วยตัวแปรชุดใหม่ $z_i\in{-1, 1}$ ผ่าน

$$
x_i = \frac{1-z_i}{2}.
$$

ที่นี่จะเห็นว่าถ้า $x_i$ เป็น $0$ แล้ว $z_i$ ต้องเป็น $1$ เมื่อแทน $x_i$ ด้วย $z_i$ ในปัญหาการหาค่าเหมาะสม ($x^TQx$) จะได้การกำหนดนิยามที่เทียบเท่ากัน

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

ถ้าเรากำหนด $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$ แล้วตัดพจน์นำหน้าและพจน์คงที่ $n^2$ ออก เราจะได้การกำหนดนิยามที่เทียบเท่าสองแบบของปัญหาการหาค่าเหมาะสมเดียวกัน

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

ที่นี่ $b$ ขึ้นอยู่กับ $Q$ โปรดทราบว่าในการหา $z^TQz + b^Tz$ เราตัดตัวประกอบ 1/4 และค่าคงที่ offset ที่เป็น $n^2$ ออก เนื่องจากไม่มีบทบาทในการหาค่าเหมาะสม

ตอนนี้เพื่อหาการกำหนดนิยามควอนตัมของปัญหา ให้ยกระดับตัวแปร $z_i$ ให้เป็นเมทริกซ์ Pauli $Z$ ซึ่งเป็นเมทริกซ์ขนาด $2\times 2$ ในรูปแบบ

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

เมื่อแทนเมทริกซ์เหล่านี้ในปัญหาการหาค่าเหมาะสมข้างต้น จะได้ Hamiltonian ดังนี้

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*นอกจากนี้โปรดจำไว้ว่าเมทริกซ์ $Z$ ถูกฝังอยู่ใน computational space ของคอมพิวเตอร์ควอนตัม นั่นคือ Hilbert space ขนาด $2^n\times 2^n$ ดังนั้น พจน์อย่างเช่น $Z_iZ_j$ ควรเข้าใจว่าเป็น tensor product $Z_i\otimes Z_j$ ที่ฝังอยู่ใน Hilbert space ขนาด $2^n\times 2^n$ ตัวอย่างเช่น ในปัญหาที่มีตัวแปรการตัดสินใจห้าตัว พจน์ $Z_1Z_3$ เข้าใจได้ว่าหมายถึง $I\otimes Z_3\otimes I\otimes Z_1\otimes I$ โดยที่ $I$ คือเมทริกซ์ identity ขนาด $2\times 2$*
</details>

Hamiltonian นี้เรียกว่า **cost function Hamiltonian** มีคุณสมบัติที่ ground state สอดคล้องกับคำตอบที่ **หาค่าต่ำสุดของฟังก์ชันต้นทุน $f(x)$**
ดังนั้น เพื่อแก้ปัญหาการหาค่าเหมาะสม คุณต้องเตรียม ground state ของ $H_C$ (หรือสถานะที่มีความทับซ้อนสูงกับมัน) บนคอมพิวเตอร์ควอนตัม จากนั้นการสุ่มตัวอย่างจากสถานะนี้จะให้คำตอบของ $min~f(x)$ ด้วยความน่าจะเป็นสูง
มาพิจารณา Hamiltonian $H_C$ สำหรับปัญหา **Max-Cut** กัน ให้แต่ละจุดยอดของกราฟสัมพันธ์กับ Qubit ในสถานะ $|0\rangle$ หรือ $|1\rangle$ โดยค่าบ่งบอกว่าจุดยอดอยู่ในกลุ่มใด เป้าหมายของปัญหาคือการหาค่าสูงสุดของจำนวนเส้นเชื่อม $(v_1, v_2)$ ที่ $v_1 = |0\rangle$ และ $v_2 = |1\rangle$ หรือในทางกลับกัน ถ้าเราเชื่อม operator $Z$ กับแต่ละ Qubit โดยที่

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

แล้วเส้นเชื่อม $(v_1, v_2)$ อยู่ในการตัดถ้า eigenvalue ของ $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$ กล่าวคือ Qubit ที่สัมพันธ์กับ $v_1$ และ $v_2$ แตกต่างกัน ในทำนองเดียวกัน $(v_1, v_2)$ ไม่อยู่ในการตัดถ้า eigenvalue ของ $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$ โปรดทราบว่าเราไม่สนใจสถานะ Qubit ที่แน่นอนของแต่ละจุดยอด แต่สนใจเพียงว่ามันเหมือนกันหรือไม่ข้ามเส้นเชื่อม ปัญหา Max-Cut ต้องการให้เราหาการกำหนดค่า Qubit บนจุดยอดที่ทำให้ eigenvalue ของ Hamiltonian ต่อไปนี้มีค่าต่ำสุด
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

กล่าวอีกนัยหนึ่ง $b_i = 0$ สำหรับทุก $i$ ในปัญหา Max-Cut ค่า $Q_{ij}$ แทนน้ำหนักของเส้นเชื่อม ในบทแนะนำนี้เราพิจารณากราฟที่ไม่มีน้ำหนัก นั่นคือ $Q_{ij} = 1.0$ สำหรับทุก $i, j$

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Hamiltonian &rarr; quantum circuit

The Hamiltonian $H_C$ contains the quantum definition of your problem. Now you can create a quantum circuit that will help *sample* good solutions from the quantum computer. The QAOA is inspired by quantum annealing and applies alternating layers of operators in the quantum circuit.

The general idea is to start in the ground state of a known system, $H^{\otimes n}|0\rangle$ above, and then steer the system into the ground state of the cost operator that you are interested in. This is done by applying the operators $\exp\{-i\gamma_k H_C\}$ and $\exp\{-i\beta_k H_m\}$ with angles $\gamma_1,...,\gamma_p$ and $\beta_1,...,\beta_p~$.


The quantum circuit that you generate is **parametrized** by $\gamma_i$ and $\beta_i$, so you can try out different values of $\gamma_i$ and $\beta_i$ and sample from the resulting state.

![Circuit diagram with QAOA layers](../docs/images/tutorials/quantum-approximate-optimization-algorithm/circuit-diagram.svg)


In this case, you will try an example with one QAOA layer that contains two parameters: $\gamma_1$ and $\beta_1$.

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

The circuit above contains a series of abstractions useful to think about quantum algorithms, but not possible to run on the hardware. To be able to run on a QPU, the circuit needs to undergo a series of operations that make up the **transpilation** or **circuit optimization** step of the pattern.

The Qiskit library offers a series of **transpilation passes** that cater to a wide range of circuit transformations. You need to make sure that your circuit is **optimized** for your purpose.

Transpilation may involves several steps, such as:

* **Initial mapping** of the qubits in the circuit (such as decision variables) to physical qubits on the device.
* **Unrolling** of the instructions in the quantum circuit to the hardware-native instructions that the backend understands.
* **Routing** of any qubits in the circuit that interact to physical qubits that are adjacent with one another.
* **Error suppression** by adding single-qubit gates to suppress noise with dynamical decoupling.


More information about transpilation is available in our [documentation](/docs/guides/transpile).

The following code transforms and optimizes the abstract circuit into a format that is ready for execution on one of devices accessible through the cloud using the **Qiskit IBM Runtime service**.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('test_heron_pok_1')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

In the QAOA workflow, the optimal QAOA parameters are found in an iterative optimization loop, which runs a series of circuit evaluations and uses a classical optimizer to find the optimal $\beta_k$ and $\gamma_k$ parameters. This execution loop is executed via the following steps:

1. Define the initial parameters
2. Instantiate a new `Session` containing the optimization loop and the primitive used to sample the circuit
3. Once an optimal set of parameters is found, execute the circuit a final time to obtain a final distribution which will be used in the post-process step.

#### Define circuit with initial parameters
We start with arbitrary chosen parameters.

In [7]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

### ขั้นตอนที่ 2: ปรับปรุงปัญหาสำหรับการรันบนฮาร์ดแวร์ควอนตัม
Circuit ข้างต้นประกอบด้วยชั้น abstraction ที่มีประโยชน์สำหรับการคิดเกี่ยวกับอัลกอริทึมควอนตัม แต่ไม่สามารถรันบนฮาร์ดแวร์ได้ เพื่อให้สามารถรันบน QPU Circuit จำเป็นต้องผ่านชุดของการดำเนินการที่ประกอบเป็นขั้นตอน **transpilation** หรือ **การปรับปรุง Circuit** ของ pattern

ไลบรารี Qiskit มี **transpilation passes** หลายชุดที่รองรับการแปลง Circuit ในหลากหลายรูปแบบ คุณต้องตรวจสอบให้แน่ใจว่า Circuit ของคุณได้รับการ **ปรับปรุง** ตามวัตถุประสงค์

Transpilation อาจเกี่ยวข้องกับหลายขั้นตอน เช่น:

* **การแมปเบื้องต้น** ของ Qubit ใน Circuit (เช่น ตัวแปรการตัดสินใจ) ไปยัง Qubit ทางกายภาพบนอุปกรณ์
* **การคลาย** คำสั่งใน Circuit ควอนตัมให้เป็นคำสั่งที่ฮาร์ดแวร์รองรับโดยตรง
* **การเดินสาย** ของ Qubit ใด ๆ ใน Circuit ที่มีปฏิสัมพันธ์กันไปยัง Qubit ทางกายภาพที่อยู่ติดกัน
* **การลดข้อผิดพลาด** โดยการเพิ่ม Gate Qubit เดี่ยวเพื่อลดสัญญาณรบกวนด้วย dynamical decoupling

ข้อมูลเพิ่มเติมเกี่ยวกับ transpilation มีอยู่ใน [เอกสารของเรา](/guides/transpile)

โค้ดต่อไปนี้แปลงและปรับปรุง Circuit แบบ abstract ให้อยู่ในรูปแบบที่พร้อมสำหรับการรันบนอุปกรณ์ที่เข้าถึงได้ผ่านคลาวด์โดยใช้ **Qiskit IBM Runtime service**

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.6295230263157894
       x: [ 1.530e+00  1.439e+00  4.071e+00  4.434e+00]
    nfev: 26
   maxcv: 0.0


![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### ขั้นตอนที่ 3: รันโดยใช้ Qiskit primitives
ใน workflow ของ QAOA พารามิเตอร์ QAOA ที่เหมาะสมจะถูกหาในลูปการหาค่าเหมาะสมแบบวนซ้ำ ซึ่งรันชุดของการประเมิน Circuit และใช้ตัวหาค่าเหมาะสมแบบคลาสสิกเพื่อหาพารามิเตอร์ $\beta_k$ และ $\gamma_k$ ที่เหมาะสมที่สุด ลูปการรันนี้ดำเนินการผ่านขั้นตอนต่อไปนี้:

1. กำหนดพารามิเตอร์เริ่มต้น
2. สร้าง `Session` ใหม่ที่ประกอบด้วยลูปการหาค่าเหมาะสมและ primitive ที่ใช้ในการสุ่มตัวอย่าง Circuit
3. เมื่อพบชุดพารามิเตอร์ที่เหมาะสมที่สุด ให้รัน Circuit อีกครั้งเป็นครั้งสุดท้ายเพื่อรับ distribution สุดท้ายที่จะใช้ในขั้นตอนการประมวลผลหลัง
#### กำหนด Circuit พร้อมพารามิเตอร์เริ่มต้น
เราเริ่มต้นด้วยพารามิเตอร์ที่เลือกโดยพลการ

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### กำหนด backend และ execution primitive
ใช้ **Qiskit Runtime primitives** เพื่อโต้ตอบกับ backend ของ IBM&reg; Primitive ทั้งสองคือ Sampler และ Estimator และการเลือก primitive ขึ้นอยู่กับประเภทของการวัดที่ต้องการรันบนคอมพิวเตอร์ควอนตัม สำหรับการหาค่าต่ำสุดของ $H_C$ ให้ใช้ Estimator เนื่องจากการวัดฟังก์ชันต้นทุนเป็นเพียงค่าความคาดหวัง $\langle H_C \rangle$
#### รัน
Primitive มี [execution modes](/guides/execution-modes) หลากหลายรูปแบบสำหรับจัดตาราง workload บนอุปกรณ์ควอนตัม และ workflow ของ QAOA รันแบบวนซ้ำใน session

![ภาพประกอบที่แสดงพฤติกรรมของโหมด runtime Single job, Batch และ Session](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

คุณสามารถเสียบฟังก์ชันต้นทุนที่ใช้ Sampler เข้าในโปรแกรมหาค่าต่ำสุดของ SciPy เพื่อหาพารามิเตอร์ที่เหมาะสม

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{28: 0.0328, 11: 0.0343, 2: 0.0296, 25: 0.0308, 16: 0.0303, 27: 0.0302, 13: 0.0323, 7: 0.0312, 4: 0.0296, 9: 0.0295, 26: 0.0321, 30: 0.031, 23: 0.0324, 31: 0.0303, 21: 0.0335, 15: 0.0317, 12: 0.0309, 29: 0.0297, 3: 0.0313, 5: 0.0312, 6: 0.0274, 10: 0.0329, 22: 0.0353, 0: 0.0315, 20: 0.0326, 8: 0.0322, 14: 0.0306, 17: 0.0295, 18: 0.0279, 1: 0.0325, 24: 0.0334, 19: 0.0295}


### Step 4: Post-process and return result in desired classical format

The post-processing step interprets the sampling output to return a solution for your original problem. In this case, you are interested in the bitstring with the highest probability as this determines the optimal cut. The symmetries in the problem allow for four possible solutions, and the sampling process will return one of them with a slightly higher probability, but you can see in the plotted distribution below that four of the bitstrings are distinctively more likely than the rest.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 1, 1, 0, 1]


In [14]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![ผลลัพธ์ของ code cell ก่อนหน้า](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

เมื่อพบพารามิเตอร์ที่เหมาะสมที่สุดสำหรับ Circuit แล้ว คุณสามารถกำหนดพารามิเตอร์เหล่านี้และสุ่มตัวอย่าง distribution สุดท้ายที่ได้จากพารามิเตอร์ที่ปรับแต่งแล้ว นี่คือจุดที่ควรใช้ primitive *Sampler* เนื่องจากเป็น probability distribution ของการวัด bitstring ที่สอดคล้องกับการตัดที่เหมาะสมของกราฟ

**หมายเหตุ:** นี่หมายถึงการเตรียมสถานะควอนตัม $\psi$ ในคอมพิวเตอร์แล้วทำการวัด การวัดจะยุบสถานะให้เป็น computational basis state เดียว เช่น `010101110000...` ซึ่งสอดคล้องกับคำตอบที่เป็นตัวเลือก $x$ ของปัญหาการหาค่าเหมาะสมเริ่มต้น ($\max f(x)$ หรือ $\min f(x)$ ขึ้นอยู่กับงาน)

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

And calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


## Part II. Scale it up!

You have access to many devices with over 100 qubits on IBM Quantum&reg; Platform. Select one on which to solve Max-Cut on a 100-node weighted graph. This is a "utility-scale" problem. The steps to build the workflow are followed the same as above, but with a much larger graph.

In [17]:
n = 100  # Number of nodes in graph
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n and edge[1] < n:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)
draw_graph(graph_100, node_size=200, with_labels=True, width=1)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif" alt="Output of the previous code cell" />

### ขั้นตอนที่ 4: ประมวลผลหลังการรันและคืนผลลัพธ์ในรูปแบบคลาสสิกที่ต้องการ
ขั้นตอนการประมวลผลหลังการรันจะตีความผลลัพธ์จากการสุ่มตัวอย่างเพื่อคืนคำตอบสำหรับปัญหาเดิม ในกรณีนี้เราสนใจ bitstring ที่มีความน่าจะเป็นสูงสุด เพราะมันจะกำหนดการตัดที่เหมาะสมที่สุด ความสมมาตรในปัญหาทำให้มีคำตอบที่เป็นไปได้สี่แบบ และกระบวนการสุ่มตัวอย่างจะคืนหนึ่งในนั้นด้วยความน่าจะเป็นที่สูงกว่าเล็กน้อย แต่จะเห็นได้จากการกระจายที่พล็อตด้านล่างว่า bitstring สี่ตัวมีความน่าจะเป็นสูงกว่าตัวที่เหลืออย่างชัดเจน

In [18]:
max_cut_paulis_100 = build_max_cut_paulis(graph_100)

cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(max_cut_paulis_100, 100)
print("Cost Function Hamiltonian:", cost_hamiltonian_100)

Cost Function Hamiltonian: SparsePauliOp(['...'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0

#### Hamiltonian &rarr; quantum circuit

In [19]:
circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

circuit_100.draw("mpl", fold=False, scale=0.2, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum execution
To scale the circuit optimization step to utility-scale problems, you can take advantage of the high performance transpilation strategies introduced in Qiskit SDK v1.0. Other tools include the new transpiler service with [AI enhanced transpiler passes](/docs/guides/ai-transpiler-passes).

In [20]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit_100 = pm.run(circuit_100)
candidate_circuit_100.draw("mpl", fold=False, scale=0.1, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3a14e7ad-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### แสดงผลการตัดที่ดีที่สุด
จาก bit string ที่เหมาะสมที่สุด เราสามารถแสดงภาพการตัดนี้บนกราฟเดิมได้

In [21]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)

    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.9939191365979383
       x: [ 1.571e+00  3.142e+00]
    nfev: 29
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

และคำนวณค่าของการตัด:

In [22]:
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)
optimized_circuit_100.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/1c432c2e-0.avif" alt="Output of the previous code cell" />

Finally, execute the circuit with the optimal parameters to sample from the corresponding distribution.

In [23]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"


pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

## ส่วนที่ 2: ขยายขนาดขึ้น!
คุณมีสิทธิ์เข้าถึงอุปกรณ์หลายตัวที่มี Qubit มากกว่า 100 ตัวบน IBM Quantum&reg; Platform เลือกตัวหนึ่งเพื่อแก้ปัญหา Max-Cut บนกราฟแบบถ่วงน้ำหนัก 100 โหนด นี่คือปัญหาระดับ "utility-scale" ขั้นตอนการสร้าง workflow จะเหมือนกับข้างต้น แต่ใช้กราฟที่ใหญ่กว่ามาก

In [24]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/0fda3611-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif)

### ขั้นตอนที่ 1: แมปอินพุตคลาสสิกไปยังปัญหาควอนตัม
#### กราฟ &rarr; Hamiltonian
ก่อนอื่น แปลงกราฟที่ต้องการแก้ไขโดยตรงเป็น Hamiltonian ที่เหมาะสมสำหรับ QAOA

In [25]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Utility for the evaluation of the expectation value of a measured state."""
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Find solution with lowest cost"""
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        # Qiskit use little endian hence the [::-1]
        candidate_sol = int(bit_str)
        # fval = qp.objective.evaluate(candidate_sol)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_sol = candidate_sol

    return min_sol


best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

Result bitstring: [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


Next, visualize the cut. Nodes of the same color belong to the same group.

In [26]:
plot_result(graph_100, best_sol_bitstring_100)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/b4a25e28-0.avif" alt="Output of the previous code cell" />

#### Hamiltonian &rarr; quantum circuit

In [27]:
cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

The value of the cut is: 124


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif)

### ขั้นตอนที่ 2: ปรับปัญหาให้เหมาะสมสำหรับการรันบนควอนตัม
เพื่อขยายขั้นตอนการปรับแต่ง Circuit ให้รองรับปัญหาระดับ utility-scale เราสามารถใช้ประโยชน์จากกลยุทธ์การ transpile ประสิทธิภาพสูงที่แนะนำใน Qiskit SDK v1.0 เครื่องมืออื่นๆ ได้แก่ transpiler service ใหม่พร้อม [AI enhanced transpiler passes](/guides/ai-transpiler-passes)

In [28]:
# auxiliary function to help plot cumulative distribution functions
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


# auxiliary function to convert bit-strings to objective values
def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""

    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [29]:
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)

Finally, you can plot the cumulative distribution function to visualize how each sample contributes to the total probability distribution and the corresponding objective value. The horizontal spread shows the range of objective values of the samples in the final distribution. Ideally, you would see that the cumulative distribution function has "jumps" at the lower end of the objective function value axis. This would mean that few solutions with low cost have high probability of being sampled. A smooth, wide curve indicates that each sample is similarly likely, and they can have very different objective values, low or high.

In [30]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, "Eagle device")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/4381a2b3-0.avif" alt="Output of the previous code cell" />

เมื่อได้พารามิเตอร์ที่เหมาะสมจากการรัน QAOA บนอุปกรณ์แล้ว ให้กำหนดพารามิเตอร์เหล่านั้นให้กับ Circuit